# Vision CISA · Reentrenamiento en Google Colab

Este notebook **no tiene lógica propia**: clona el repositorio y llama a
`python -m reentrenamiento`. Así lo que se hace en Colab es lo mismo que está
probado en el repo (`tests/`). Guía: `docs/PIPELINE_REENTRENAMIENTO.md`.

**Antes:** `Entorno de ejecución → Cambiar tipo → GPU` · inicia sesión con la
cuenta del proyecto · en Drive, la carpeta `CISA_Reentrenamiento/` con:
`modelos/<modelo>_v3.pt`, `M3/crudas/` (recortes) y `padron.csv`.
Para bajar datasets de Roboflow por código: secreto 🔑 `ROBOFLOW_API_KEY`
(nunca pegues llaves en el código).

| Parte | Qué | Dónde trabaja una persona |
|---|---|---|
| A | Fotogramas clave + pre-etiquetas | — |
| — | Corregir el lote | **Roboflow** |
| B | Validar, dividir, entrenar | — |
| C | Evaluar v4 contra v3 → ACEPTAR/RECHAZAR | Revisar el reporte |

In [ ]:
# 0) Configuración — ajusta solo esta celda
MODELO   = "m3"                       # m1 | m2 | m3
VERSION  = "v4"
RAMA     = "mejora/reentrenamiento-eficiente"
BASE     = "/content/drive/MyDrive/CISA_Reentrenamiento"
PESOS_V3 = f"{BASE}/modelos/modelo_digitos_v3.pt"
CRUDAS   = f"{BASE}/M3/crudas"         # carpeta con los recortes sin etiquetar
PADRON   = f"{BASE}/padron.csv"
LOTE     = f"{BASE}/M3/lote_{VERSION}"
CONGELADO = f"{BASE}/M3/congelado_m3.txt"   # set de prueba fijo (se crea la 1a vez)
IMGSZ    = 640                        # M2 = 1280

In [ ]:
# 1) GPU, Drive, repo y dependencias
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')
!rm -rf /content/vision-cisa && git clone -q -b {RAMA} https://github.com/visioncisa/vision-cisa.git /content/vision-cisa
%cd /content/vision-cisa
!pip install -q -r requirements-reentrenamiento.txt ultralytics==8.4.171 roboflow
!python -m reentrenamiento --help | head -15

## A) Fotogramas clave + pre-etiquetas  →  subir el lote a Roboflow

In [ ]:
!python -m reentrenamiento preanotar --entrada "{CRUDAS}" --modelo {MODELO} --pesos "{PESOS_V3}" \
    --padron "{PADRON}" --salida "{LOTE}" --imgsz {IMGSZ}
import pandas as pd
r = pd.read_csv(f"{LOTE}/resumen.csv")
display(r[r.decision != "descartar"].head(20))

**Ahora en Roboflow:** proyecto M3 → *Upload Data* → arrastra la carpeta del lote
(desde Drive). Corrige empezando por las etiquetas `sistema_distinto` y `desacuerdo`.
Reglas: `docs/POLITICA_DATASET.md`. Al terminar: *Generate* → nueva versión
**sin aumentos que volteen** (M3) → anota el número de versión abajo.

## B) Dataset → validar → dividir → entrenar

In [ ]:
from google.colab import userdata
from roboflow import Roboflow
WORKSPACE, PROYECTO, VERSION_RF = "cisas-workspace", "m3-rzzw2", 1    # ← versión generada en Roboflow
rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
ds = rf.workspace(WORKSPACE).project(PROYECTO).version(VERSION_RF).download("yolov8", location="/content/rf_export")

In [ ]:
# Roboflow exporta ya dividido; se junta y se vuelve a dividir POR PASADA con el test congelado
!mkdir -p /content/plano/images /content/plano/labels
!cp /content/rf_export/*/images/* /content/plano/images/ && cp /content/rf_export/*/labels/* /content/plano/labels/
!cp /content/rf_export/data.yaml /content/plano/data.yaml
!python -m reentrenamiento validar --dataset /content/plano --modelo {MODELO} --sin-fuga
!python -m reentrenamiento dividir --origen /content/plano --destino /content/datasets/{MODELO}_{VERSION} \
    --modelo {MODELO} --congelado "{CONGELADO}"
!python -m reentrenamiento validar --dataset /content/datasets/{MODELO}_{VERSION} --modelo {MODELO}

In [ ]:
# Entrenamiento (fine-tuning desde v3). Edita experimentos/m3_v4_finetune.yaml para cambiar args.
!python -m reentrenamiento entrenar --experimento experimentos/m3_v4_finetune.yaml \
    --registro "{BASE}/experimentos/registro.jsonl" --proyecto "{BASE}/experimentos/corridas"

## C) Evaluar v4 contra v3 en el set congelado

In [ ]:
import glob
CANDIDATO = sorted(glob.glob(f"{BASE}/experimentos/corridas/*/weights/best.pt"))[-1]
print("Candidato:", CANDIDATO)
!python -m reentrenamiento evaluar --verdad /content/datasets/{MODELO}_{VERSION}/test --modelo {MODELO} \
    --pesos-actual "{PESOS_V3}" --pesos-candidato "{CANDIDATO}" \
    --data-yaml /content/datasets/{MODELO}_{VERSION}/data.yaml --imgsz {IMGSZ} \
    --salida "{BASE}/experimentos/eval_{MODELO}_{VERSION}" --titulo "{MODELO} {VERSION} vs v3"
from IPython.display import Markdown, display
display(Markdown(open(f"{BASE}/experimentos/eval_{MODELO}_{VERSION}/evaluacion.md").read()))

**Si dice ACEPTAR:** el `best.pt` va a la Jetson y se compila allá con
`scripts/exportar_engine.sh` (ver `docs/DESPLIEGUE.md`). **Nunca** se copia un
`.engine` hecho en Colab: TensorRT solo es válido en el equipo donde se compila.

**Si dice RECHAZAR:** el reporte dice por qué (olvido de una clase, baja
la exactitud…). Se corrigen etiquetas o se agregan casos y se repite B.